# FlowThought PoC

**Flow Matching for Reasoning in LLM Hidden State Space**

This notebook implements the FlowThought proof-of-concept:
1. **Phase 1** — Extract hidden states from Qwen2.5-0.5B-Instruct on GSM8K
2. **Phase 2** — Train an OT-CFM velocity field (VelocityMLP) to map noise → CoT hidden states
3. **Phase 3** — Evaluate via answer probe: compare no-CoT, real CoT, random, and FlowThought

Runs end-to-end on a free Colab T4 GPU.

In [ ]:
# Cell 1: Install dependencies (torch is pre-installed on Colab)
!pip install -q transformers==4.47.0 datasets==3.2.0 accelerate==1.2.1 matplotlib==3.9.3

In [ ]:
# Cell 2: Imports + Config
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import re
import gc
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

CONFIG = {
    "model_name": "Qwen/Qwen2.5-0.5B-Instruct",
    "hidden_dim": 896,
    "n_train": 200,             # smoke test — scale to 2000 after validation
    "n_test": 50,               # smoke test — scale to 500 after validation
    "batch_size_extract": 32,   # for short texts (problem-only)
    "batch_size_extract_long": 4,  # for long texts (problem+CoT) — T4 safe
    "max_length": 512,
    # FM training
    "fm_epochs": 50,            # smoke test — scale to 300 after validation
    "fm_batch_size": 200,       # full dataset per epoch
    "fm_lr": 3e-4,
    "fm_hidden": 1024,
    "fm_layers": 4,
    "grad_clip": 1.0,
    "fm_checkpoint_every": 25,
    # ODE
    "ode_steps": 20,
    # Probe
    "probe_epochs": 50,         # smoke test — scale to 100 after validation
    "probe_lr": 1e-3,
    "probe_hidden": 256,
    # Paths
    "cache_dir": "flowthought_cache",
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 3: Test framework
@dataclass
class TestResults:
    passed: int = 0
    failed: int = 0
    details: List[str] = field(default_factory=list)

    def check(self, condition: bool, name: str):
        if condition:
            self.passed += 1
            self.details.append(f"  PASS: {name}")
        else:
            self.failed += 1
            self.details.append(f"  FAIL: {name}")

    def summary(self, section: str):
        status = "ALL PASSED" if self.failed == 0 else f"{self.failed} FAILED"
        print(f"\n[{section}] {self.passed}/{self.passed + self.failed} tests passed — {status}")
        for d in self.details:
            print(d)
        assert self.failed == 0, f"{self.failed} tests failed in {section}"

# Quick self-test
t = TestResults()
t.check(True, "framework works")
t.summary("Self-test")

In [ ]:
# Cell 4: Data loading — GSM8K + answer parsing
from datasets import load_dataset

ds = load_dataset("openai/gsm8k", "main")
print(f"Train: {len(ds['train'])}, Test: {len(ds['test'])}")

def parse_answer(answer_str: str) -> Optional[float]:
    """Extract the numeric answer after #### from GSM8K format."""
    match = re.search(r"####\s*([\-\d,\.]+)", answer_str)
    if match:
        return float(match.group(1).replace(",", ""))
    return None

def get_cot_and_answer(example):
    """Split GSM8K answer field into CoT reasoning and numeric answer."""
    text = example["answer"]
    parts = text.split("####")
    cot = parts[0].strip()
    ans = parse_answer(text)
    return cot, ans

# Tests
t = TestResults()
t.check(parse_answer("blah blah\n#### 42") == 42.0, "parse simple")
t.check(parse_answer("stuff\n#### 1,234") == 1234.0, "parse comma")
t.check(parse_answer("stuff\n#### -5") == -5.0, "parse negative")
t.check(parse_answer("no answer here") is None, "parse missing")

cot, ans = get_cot_and_answer(ds["train"][0])
t.check(isinstance(cot, str) and len(cot) > 10, "cot is string")
t.check(isinstance(ans, float), "answer is float")
t.summary("Data Loading")

print(f"\nExample question: {ds['train'][0]['question'][:100]}...")
print(f"Answer: {ans}")

In [ ]:
# Cell 5: Model loading — SDPA attention + explicit device placement
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # left-pad for causal LM

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=torch.float16,
    attn_implementation="sdpa",  # use PyTorch SDPA for faster attention
).to(device)
model.eval()

# Register forward hook on the final layernorm (after all transformer layers)
# This gives us the same output as hidden_states[-1] from output_hidden_states=True
_last_hidden = {}
def _hook_fn(module, input, output):
    _last_hidden["val"] = output

_hook_handle = model.model.norm.register_forward_hook(_hook_fn)

# Test
t = TestResults()
test_input = tokenizer("Hello", return_tensors="pt").to(device)
with torch.no_grad():
    _ = model(**test_input)
last_hidden = _last_hidden["val"]
t.check(last_hidden.shape[-1] == CONFIG["hidden_dim"], f"hidden dim = {last_hidden.shape[-1]}")
t.check(not torch.isnan(last_hidden).any(), "no NaN in hidden states")
t.summary("Model Loading")
print(f"Model loaded: {CONFIG['model_name']} (SDPA + hook-based extraction)")

In [ ]:
# Cell 6: Hidden state extraction functions (hook-based, no output_hidden_states overhead)

@torch.no_grad()
def extract_hidden_states(texts: List[str], batch_size: int = 32) -> torch.Tensor:
    """Extract last-layer, last-token hidden states via forward hook.
    Returns: (N, hidden_dim) float32 tensor on CPU.
    """
    all_states = []
    n_batches = (len(texts) + batch_size - 1) // batch_size
    n_truncated = 0
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(
            batch, return_tensors="pt", padding=True, truncation=True,
            max_length=CONFIG["max_length"],
        ).to(device)
        # Count truncated examples
        for j in range(len(batch)):
            if inputs["attention_mask"][j].sum() == CONFIG["max_length"]:
                n_truncated += 1
        # Forward pass — hook captures last layer output
        _ = model(**inputs)
        hidden = _last_hidden["val"]  # (B, seq_len, hidden_dim)
        # With left-padding, last token is always the last position
        last_token_states = hidden[:, -1, :]  # (B, hidden_dim)
        all_states.append(last_token_states.float().cpu())
        if (i // batch_size) % 25 == 0:
            print(f"    batch {i // batch_size + 1}/{n_batches}")
    if n_truncated > 0:
        print(f"    WARNING: {n_truncated}/{len(texts)} examples truncated at max_length={CONFIG['max_length']}")
    return torch.cat(all_states, dim=0)

# Tests
t = TestResults()
test_texts = ["What is 2+2?", "The answer to 2+2 is 4 because addition."]
test_states = extract_hidden_states(test_texts, batch_size=2)
t.check(test_states.shape == (2, CONFIG["hidden_dim"]), f"shape: {test_states.shape}")
t.check(not torch.isnan(test_states).any(), "no NaN")
t.check(not torch.isinf(test_states).any(), "no Inf")
cos_sim = F.cosine_similarity(test_states[0:1], test_states[1:2]).item()
t.check(cos_sim < 0.99, f"states are distinct (cos_sim={cos_sim:.4f})")
t.summary("Hidden State Extraction")

In [ ]:
# Cell 7: Phase 1 — Run extraction on GSM8K (with caching) + free LLM after

cache_dir = Path(CONFIG["cache_dir"])
cache_dir.mkdir(exist_ok=True)

def run_extraction(split, n_examples):
    """Extract condition (problem) and target (problem+CoT) hidden states."""
    cache_file = cache_dir / f"{split}_{n_examples}.pt"
    if cache_file.exists():
        print(f"Loading cached {split} data from {cache_file}")
        return torch.load(cache_file, weights_only=False)

    data = ds[split].select(range(min(n_examples, len(ds[split]))))
    problems = []
    problems_with_cot = []
    answers = []
    skipped = 0

    for ex in data:
        q = ex["question"]
        cot, ans = get_cot_and_answer(ex)
        if ans is None:
            skipped += 1
            continue
        problems.append(q)
        problems_with_cot.append(f"{q}\n{cot}")
        answers.append(ans)

    if skipped > 0:
        print(f"  Skipped {skipped} examples with unparseable answers")
    print(f"Extracting {len(problems)} {split} examples...")

    print("  Extracting condition (problem-only) hidden states...")
    h_cond = extract_hidden_states(problems, CONFIG["batch_size_extract"])

    print("  Extracting target (problem+CoT) hidden states (smaller batches for long seqs)...")
    h_target = extract_hidden_states(problems_with_cot, CONFIG["batch_size_extract_long"])

    answers_tensor = torch.tensor(answers, dtype=torch.float32)

    result = {"h_cond": h_cond, "h_target": h_target, "answers": answers_tensor}
    torch.save(result, cache_file)
    print(f"  Saved to {cache_file}")
    return result

train_data = run_extraction("train", CONFIG["n_train"])
test_data = run_extraction("test", CONFIG["n_test"])

# Free LLM from GPU — it's not needed for Phase 2/3
_hook_handle.remove()
del model, tokenizer, _last_hidden
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()
    print(f"\nLLM freed. GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
else:
    print("\nLLM freed from memory.")

# z-score normalization (fit on train)
h_mean = train_data["h_target"].mean(dim=0)
h_std = train_data["h_target"].std(dim=0).clamp(min=1e-6)
a_mean = train_data["answers"].mean()
a_std = train_data["answers"].std().clamp(min=1e-6)

def normalize_h(h):
    return (h - h_mean) / h_std

def normalize_a(a):
    return (a - a_mean) / a_std

def denormalize_a(a):
    return a * a_std + a_mean

# Tests
t = TestResults()
for name, d in [("train", train_data), ("test", test_data)]:
    t.check(d["h_cond"].shape[1] == CONFIG["hidden_dim"], f"{name} cond dim")
    t.check(d["h_target"].shape[1] == CONFIG["hidden_dim"], f"{name} target dim")
    t.check(not torch.isnan(d["h_cond"]).any(), f"{name} cond no NaN")
    t.check(not torch.isnan(d["h_target"]).any(), f"{name} target no NaN")
    cos = F.cosine_similarity(d["h_cond"], d["h_target"]).mean().item()
    t.check(cos < 0.99, f"{name} cond != target (mean cos={cos:.4f})")

t.check(train_data["h_cond"].shape[0] >= CONFIG["n_train"] * 0.9, "enough train examples")
t.check(test_data["h_cond"].shape[0] >= CONFIG["n_test"] * 0.9, "enough test examples")
t.summary("Phase 1: Extraction")

print(f"\nTrain: {train_data['h_cond'].shape[0]} examples")
print(f"Test: {test_data['h_cond'].shape[0]} examples")
print(f"Hidden dim: {CONFIG['hidden_dim']}")

In [ ]:
# Cell 8: VelocityMLP definition

class VelocityMLP(nn.Module):
    """MLP velocity field for OT-CFM: v_theta(h_t, t, c) -> dh/dt"""

    def __init__(self, hidden_dim, mlp_hidden, n_layers):
        super().__init__()
        # Input: h_t (hidden_dim) + t (1) + c (hidden_dim)
        input_dim = hidden_dim * 2 + 1
        layers = []
        layers.append(nn.Linear(input_dim, mlp_hidden))
        layers.append(nn.SiLU())
        for _ in range(n_layers - 1):
            layers.append(nn.Linear(mlp_hidden, mlp_hidden))
            layers.append(nn.LayerNorm(mlp_hidden))
            layers.append(nn.SiLU())
        layers.append(nn.Linear(mlp_hidden, hidden_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, h_t, t, c):
        """
        h_t: (B, hidden_dim) — interpolated state
        t:   (B, 1) — time
        c:   (B, hidden_dim) — condition (problem encoding)
        Returns: (B, hidden_dim) — predicted velocity
        """
        x = torch.cat([h_t, t, c], dim=-1)
        return self.net(x)

# Tests
t = TestResults()
test_mlp = VelocityMLP(CONFIG["hidden_dim"], CONFIG["fm_hidden"], CONFIG["fm_layers"]).to(device)
B = 4
h_t = torch.randn(B, CONFIG["hidden_dim"], device=device)
t_in = torch.rand(B, 1, device=device)
c = torch.randn(B, CONFIG["hidden_dim"], device=device)
out = test_mlp(h_t, t_in, c)
t.check(out.shape == (B, CONFIG["hidden_dim"]), f"output shape: {out.shape}")
t.check(not torch.isnan(out).any(), "no NaN")
t.check(not torch.isinf(out).any(), "no Inf")
n_params = sum(p.numel() for p in test_mlp.parameters())
t.check(n_params > 0, f"has {n_params:,} parameters")
t.summary("VelocityMLP")
del test_mlp

In [ ]:
# Cell 9: CFM loss function (with logit-normal time sampling)

def cfm_loss(velocity_net, h_target, h_cond):
    """
    Conditional Flow Matching loss with logit-normal time sampling.
    - Source: h_0 ~ N(0, I)
    - Target: h_1 = normalized CoT hidden states
    - Condition: c = normalized problem hidden states
    - Linear interpolation: h_t = (1-t)*h_0 + t*h_1
    - True velocity: u_t = h_1 - h_0
    - Loss: MSE(v_theta(h_t, t, c), u_t)

    Uses logit-normal time sampling (Meta Flow Matching Guide, 2024) which
    biases training toward intermediate timesteps where velocity is hardest.
    """
    B = h_target.shape[0]
    h_0 = torch.randn_like(h_target)  # source noise

    # Logit-normal time sampling: t = sigmoid(N(0, 1)), biases toward t=0.5
    t = torch.sigmoid(torch.randn(B, 1, device=h_target.device))

    # Linear interpolation
    h_t = (1 - t) * h_0 + t * h_target

    # True velocity (OT path)
    u_t = h_target - h_0

    # Predicted velocity
    v_pred = velocity_net(h_t, t, h_cond)

    return F.mse_loss(v_pred, u_t)

# Test: loss decreases over a few steps
t = TestResults()
test_net = VelocityMLP(CONFIG["hidden_dim"], 256, 2).to(device)
test_opt = torch.optim.Adam(test_net.parameters(), lr=1e-3)

h_tgt_small = normalize_h(train_data["h_target"][:32]).to(device)
h_cnd_small = normalize_h(train_data["h_cond"][:32]).to(device)

losses = []
for step in range(20):
    test_opt.zero_grad()
    loss = cfm_loss(test_net, h_tgt_small, h_cnd_small)
    loss.backward()
    test_opt.step()
    losses.append(loss.item())

t.check(losses[-1] < losses[0], f"loss decreased: {losses[0]:.4f} -> {losses[-1]:.4f}")
t.check(not np.isnan(losses[-1]), "loss not NaN")
t.summary("CFM Loss")
del test_net, test_opt

In [ ]:
# Cell 10: Phase 2 — Train FM (AMP + torch.compile + EMA + checkpointing)
import copy

velocity_net = VelocityMLP(
    CONFIG["hidden_dim"], CONFIG["fm_hidden"], CONFIG["fm_layers"]
).to(device)

# torch.compile for kernel fusion — test with a dummy forward pass to catch lazy failures
velocity_net_compiled = velocity_net
if device.type == "cuda":
    try:
        _candidate = torch.compile(velocity_net)
        _test_out = _candidate(
            torch.randn(2, CONFIG["hidden_dim"], device=device),
            torch.rand(2, 1, device=device),
            torch.randn(2, CONFIG["hidden_dim"], device=device),
        )
        del _test_out
        velocity_net_compiled = _candidate
        print("torch.compile: enabled")
    except Exception as e:
        print(f"torch.compile: disabled ({e})")

# EMA copy for stable inference
ema_net = copy.deepcopy(velocity_net)
ema_decay = 0.999

optimizer = torch.optim.AdamW(velocity_net.parameters(), lr=CONFIG["fm_lr"], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["fm_epochs"])
scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)

# Prepare normalized data — all on GPU (only ~14MB total)
h_target_train = normalize_h(train_data["h_target"]).to(device)
h_cond_train = normalize_h(train_data["h_cond"]).to(device)

n_train = h_target_train.shape[0]
loss_history = []
checkpoint_dir = cache_dir / "fm_checkpoints"
checkpoint_dir.mkdir(exist_ok=True)

# Check for existing checkpoint to resume from
start_epoch = 0
latest_ckpt = checkpoint_dir / "latest.pt"
if latest_ckpt.exists():
    ckpt = torch.load(latest_ckpt, weights_only=False)
    velocity_net.load_state_dict(ckpt["model"])
    ema_net.load_state_dict(ckpt["ema"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    if "scaler" in ckpt:
        scaler.load_state_dict(ckpt["scaler"])
    loss_history = ckpt["loss_history"]
    start_epoch = ckpt["epoch"] + 1
    print(f"Resumed from checkpoint at epoch {start_epoch}")

print(f"Training FM: epochs {start_epoch}-{CONFIG['fm_epochs']}, {n_train} examples, batch_size={CONFIG['fm_batch_size']}")
print(f"VelocityMLP params: {sum(p.numel() for p in velocity_net.parameters()):,}")
if device.type == "cuda":
    print(f"GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

for epoch in range(start_epoch, CONFIG["fm_epochs"]):
    perm = torch.randperm(n_train, device=device)
    epoch_losses = []

    for i in range(0, n_train, CONFIG["fm_batch_size"]):
        idx = perm[i:i + CONFIG["fm_batch_size"]]
        h_tgt = h_target_train[idx]
        h_cnd = h_cond_train[idx]

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            loss = cfm_loss(velocity_net_compiled, h_tgt, h_cnd)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(velocity_net.parameters(), CONFIG["grad_clip"])
        scaler.step(optimizer)
        scaler.update()
        epoch_losses.append(loss.item())

    # EMA update
    with torch.no_grad():
        for p_ema, p_model in zip(ema_net.parameters(), velocity_net.parameters()):
            p_ema.mul_(ema_decay).add_(p_model, alpha=1 - ema_decay)

    scheduler.step()
    avg_loss = np.mean(epoch_losses)
    loss_history.append(avg_loss)

    if (epoch + 1) % 50 == 0 or epoch == start_epoch:
        print(f"  Epoch {epoch+1:3d}/{CONFIG['fm_epochs']} | Loss: {avg_loss:.6f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    # Checkpoint
    if (epoch + 1) % CONFIG["fm_checkpoint_every"] == 0:
        torch.save({
            "epoch": epoch,
            "model": velocity_net.state_dict(),
            "ema": ema_net.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "loss_history": loss_history,
        }, latest_ckpt)

# Plot
plt.figure(figsize=(8, 4))
plt.semilogy(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Loss (log)")
plt.title("FM Training Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Test
t = TestResults()
t.check(loss_history[-1] < loss_history[0], f"loss decreased: {loss_history[0]:.4f} -> {loss_history[-1]:.4f}")
t.check(loss_history[-1] < loss_history[0] * 0.5, "loss decreased by >50%")
t.summary("Phase 2: FM Training")

In [ ]:
# Cell 11: ODE integration (RK4 — same quality as Euler-50 in ~20 steps)

@torch.no_grad()
def rk4_integrate(velocity_net, h_cond, n_steps=20, device=None):
    """
    RK4-integrate the learned velocity field from t=0 to t=1.
    h_cond: (B, hidden_dim) — normalized condition vectors
    Returns: (B, hidden_dim) — generated hidden states at t=1
    """
    if device is None:
        device = h_cond.device
    B, D = h_cond.shape
    dt = 1.0 / n_steps
    h_t = torch.randn(B, D, device=device)

    for step in range(n_steps):
        t_val = step * dt
        t1 = torch.full((B, 1), t_val, device=device)
        t2 = torch.full((B, 1), t_val + dt / 2, device=device)
        t3 = torch.full((B, 1), t_val + dt, device=device)

        k1 = velocity_net(h_t, t1, h_cond)
        k2 = velocity_net(h_t + k1 * dt / 2, t2, h_cond)
        k3 = velocity_net(h_t + k2 * dt / 2, t2, h_cond)
        k4 = velocity_net(h_t + k3 * dt, t3, h_cond)

        h_t = h_t + (k1 + 2 * k2 + 2 * k3 + k4) * dt / 6

    return h_t

# Tests — use EMA net for inference (more stable)
t = TestResults()
test_cond = normalize_h(test_data["h_cond"][:16]).to(device)
generated = rk4_integrate(ema_net, test_cond, CONFIG["ode_steps"])
t.check(generated.shape == (16, CONFIG["hidden_dim"]), f"shape: {generated.shape}")
t.check(not torch.isnan(generated).any(), "no NaN")
t.check(not torch.isinf(generated).any(), "no Inf")
pairwise_cos = F.cosine_similarity(generated[:-1], generated[1:]).mean().item()
t.check(pairwise_cos < 0.99, f"not collapsed (avg pairwise cos={pairwise_cos:.4f})")
t.summary("RK4 ODE Integration")

In [ ]:
# Cell 12: Answer probe — MLP trained to predict answer from hidden states

class AnswerProbe(nn.Module):
    def __init__(self, hidden_dim, probe_hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, probe_hidden),
            nn.ReLU(),
            nn.Linear(probe_hidden, probe_hidden),
            nn.ReLU(),
            nn.Linear(probe_hidden, 1),
        )

    def forward(self, h):
        return self.net(h).squeeze(-1)

# Train probe on real CoT hidden states
probe = AnswerProbe(CONFIG["hidden_dim"], CONFIG["probe_hidden"]).to(device)
probe_opt = torch.optim.Adam(probe.parameters(), lr=CONFIG["probe_lr"])

h_target_norm = normalize_h(train_data["h_target"]).to(device)
a_norm = normalize_a(train_data["answers"]).to(device)

probe_losses = []
for epoch in range(CONFIG["probe_epochs"]):
    perm = torch.randperm(h_target_norm.shape[0], device=device)
    epoch_loss = []
    for i in range(0, h_target_norm.shape[0], 128):
        idx = perm[i:i+128]
        pred = probe(h_target_norm[idx])
        loss = F.mse_loss(pred, a_norm[idx])
        probe_opt.zero_grad()
        loss.backward()
        probe_opt.step()
        epoch_loss.append(loss.item())
    probe_losses.append(np.mean(epoch_loss))

probe.eval()

# Test on train set
with torch.no_grad():
    train_pred = denormalize_a(probe(h_target_norm)).cpu()
    train_true = train_data["answers"]
    train_corr = np.corrcoef(train_pred.numpy(), train_true.numpy())[0, 1]

t = TestResults()
t.check(probe_losses[-1] < probe_losses[0], f"probe loss decreased: {probe_losses[0]:.4f} -> {probe_losses[-1]:.4f}")
t.check(train_corr > 0.1, f"train correlation: {train_corr:.4f}")
t.summary("Answer Probe")
print(f"Probe train correlation: {train_corr:.4f}")

In [ ]:
# Cell 13: Full evaluation — 4 baselines (using EMA net + RK4)

def evaluate_method(name, h_states, true_answers):
    """Evaluate hidden states via the answer probe."""
    with torch.no_grad():
        h_norm = normalize_h(h_states).to(device)
        pred_norm = probe(h_norm).cpu()
        pred = denormalize_a(pred_norm)
    true = true_answers.numpy()
    pred_np = pred.numpy()

    corr = np.corrcoef(pred_np, true)[0, 1] if len(true) > 1 else 0.0
    exact = np.mean(np.abs(np.round(pred_np) - true) < 0.5)
    mse = np.mean((pred_np - true) ** 2)

    return {"name": name, "correlation": corr, "exact_match": exact, "mse": mse}

# Prepare test data
test_answers = test_data["answers"]
n_test = test_data["h_cond"].shape[0]

# Method 1: No-CoT (condition/problem hidden states only)
r_nocot = evaluate_method("No-CoT", test_data["h_cond"], test_answers)

# Method 2: Real CoT (ground truth target hidden states)
r_real = evaluate_method("Real CoT", test_data["h_target"], test_answers)

# Method 3: Random (Gaussian noise)
r_random = evaluate_method("Random", torch.randn(n_test, CONFIG["hidden_dim"]) * h_std + h_mean, test_answers)

# Method 4: FlowThought (FM-generated hidden states via EMA + RK4)
h_cond_test_norm = normalize_h(test_data["h_cond"]).to(device)
h_flow = rk4_integrate(ema_net, h_cond_test_norm, CONFIG["ode_steps"])
h_flow_denorm = (h_flow.cpu() * h_std + h_mean)
r_flow = evaluate_method("FlowThought", h_flow_denorm, test_answers)

results = [r_nocot, r_real, r_random, r_flow]

# Diversity analysis
def diversity_score(h):
    """Mean pairwise cosine distance (1 - cos_sim) on a subsample."""
    h_sub = h[:min(200, len(h))]
    h_norm_vec = F.normalize(h_sub.float(), dim=-1)
    sim_matrix = h_norm_vec @ h_norm_vec.T
    mask = ~torch.eye(len(h_sub), dtype=torch.bool)
    return (1 - sim_matrix[mask].mean()).item()

div_real = diversity_score(test_data["h_target"])
div_flow = diversity_score(h_flow_denorm)
div_random = diversity_score(torch.randn(n_test, CONFIG["hidden_dim"]))

# Distribution match: mean/std distance between flow and real
real_mean = normalize_h(test_data["h_target"]).mean(dim=0)
flow_mean = h_flow.cpu().mean(dim=0)
mean_dist = (real_mean - flow_mean).norm().item()

real_std = normalize_h(test_data["h_target"]).std(dim=0)
flow_std = h_flow.cpu().std(dim=0)
std_dist = (real_std - flow_std).norm().item()

print("\n" + "="*70)
print("EVALUATION RESULTS")
print("="*70)
print(f"{'Method':<15} {'Correlation':>12} {'Exact Match':>12} {'MSE':>12}")
print("-"*51)
for r in results:
    print(f"{r['name']:<15} {r['correlation']:>12.4f} {r['exact_match']:>12.4f} {r['mse']:>12.1f}")

print(f"\nDiversity (cosine distance):")
print(f"  Real CoT: {div_real:.4f}")
print(f"  FlowThought: {div_flow:.4f}")
print(f"  Random: {div_random:.4f}")

print(f"\nDistribution match (FlowThought vs Real CoT):")
print(f"  Mean distance: {mean_dist:.4f}")
print(f"  Std distance: {std_dist:.4f}")

# Tests
t = TestResults()
t.check(r_flow["correlation"] > r_random["correlation"], "FlowThought > random (correlation)")
t.check(r_real["correlation"] > r_random["correlation"], "Real CoT > random (correlation)")
t.check(div_flow > 0.01, f"FlowThought diversity > 0 ({div_flow:.4f})")
t.check(not np.isnan(r_flow["correlation"]), "FlowThought correlation not NaN")
t.summary("Full Evaluation")

In [ ]:
# Cell 14: Results summary — charts + JSON export

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

names = [r["name"] for r in results]
colors = ["#888", "#2ecc71", "#e74c3c", "#3498db"]

# Correlation
axes[0].bar(names, [r["correlation"] for r in results], color=colors)
axes[0].set_title("Correlation with True Answer")
axes[0].set_ylabel("Pearson r")
axes[0].tick_params(axis='x', rotation=15)

# Exact match
axes[1].bar(names, [r["exact_match"] for r in results], color=colors)
axes[1].set_title("Exact Match Rate")
axes[1].set_ylabel("Accuracy")
axes[1].tick_params(axis='x', rotation=15)

# Diversity
div_names = ["Real CoT", "FlowThought", "Random"]
div_vals = [div_real, div_flow, div_random]
axes[2].bar(div_names, div_vals, color=["#2ecc71", "#3498db", "#e74c3c"])
axes[2].set_title("Diversity (Cosine Distance)")
axes[2].set_ylabel("Mean Pairwise Distance")
axes[2].tick_params(axis='x', rotation=15)

plt.suptitle("FlowThought PoC Results", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Save results
summary = {
    "config": CONFIG,
    "results": results,
    "diversity": {"real": div_real, "flow": div_flow, "random": div_random},
    "distribution_match": {"mean_dist": mean_dist, "std_dist": std_dist},
    "fm_final_loss": loss_history[-1],
    "probe_train_corr": train_corr,
}

with open("flowthought_results.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print("\nResults saved to flowthought_results.json")
print("\n" + "="*70)
print("INTERPRETATION")
print("="*70)
print(f"""\n
Key findings:
- FlowThought correlation: {r_flow['correlation']:.4f} (vs random: {r_random['correlation']:.4f})
- Real CoT correlation: {r_real['correlation']:.4f} (upper bound)
- FlowThought > random confirms FM learns meaningful structure in hidden space
- Diversity {div_flow:.4f} shows FM generates varied states (not mode-collapsed)
- Distribution match (mean dist: {mean_dist:.4f}) indicates how close FM matches real CoT distribution

This PoC validates the core hypothesis: flow matching can learn to generate
reasoning-like hidden states conditioned on problem encodings. The gap between
FlowThought and Real CoT represents the opportunity for:
1. Larger velocity networks
2. More training data
3. RLVR fine-tuning (Flow-GRPO)
4. Coconut-style hidden state injection for full end-to-end evaluation
""")